# 07 - Transformer fine-tuning (GPU notebook)

This notebook is written to run on a GPU runtime (Google Colab, T4 or better). It is self-contained: it does not import from `src/`, it only needs the three partition files written by notebook 04.

Four models are fine-tuned on the raw `review_text`:

| key | model | training data |
|---|---|---|
| `xlmr` | xlm-roberta-base | full bilingual set |
| `mbert` | bert-base-multilingual-cased | full bilingual set |
| `arabert` | aubmindlab/bert-base-arabertv2 | Arabic subset |
| `marbert` | UBC-NLP/MARBERT | Arabic subset |

Settings are fixed in advance: max length 128, class-weighted cross entropy, learning rate grid {2e-5, 3e-5}, up to 3 epochs, early stopping on validation macro-F1, seed 42. Every model scores the *same* test partition so predictions pair instance by instance with the baselines.

**How to use on Colab**
1. Upload `data/processed/train.parquet`, `val.parquet`, `test.parquet` (or mount Drive and point `DATA_DIR` at the folder).
2. Run all cells. Each model writes `<key>_test.csv` and appends to `transformer_runs.json` in `OUT_DIR`.
3. Download `OUT_DIR` and copy the four `*_test.csv` files into `results/predictions/` and `transformer_runs.json` into `results/tables/` in the repo, then run notebooks 08 and 10 again.

A `SMOKE_TEST = True` flag runs a tiny version on CPU in a couple of minutes to prove the code path works.

In [ ]:
# On Colab this installs the few packages that are missing. Nothing to do locally.
import importlib, subprocess, sys
for pkg, mod in [("transformers", "transformers"), ("accelerate", "accelerate"), ("sentencepiece", "sentencepiece"),
                 ("protobuf", "google.protobuf"), ("pyarrow", "pyarrow"), ("scikit-learn", "sklearn")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

In [ ]:
import json, os, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SMOKE_TEST = False          # True -> 300 training rows, 1 epoch, a tiny model, CPU is fine
SEED = 42
MAX_LEN = 128
BATCH = 32
EPOCHS = 3
LR_GRID = [2e-5, 3e-5]
PATIENCE = 1                # epochs without val macro-F1 improvement before stopping

# where the partition files are and where outputs go
DATA_DIR = Path("/content/drive/MyDrive/uae-gov-satisfaction-nlp/data/processed") if Path("/content").exists() else Path("../data/processed")
OUT_DIR = Path("/content/drive/MyDrive/uae-gov-satisfaction-nlp/results") if Path("/content").exists() else Path("../results/transformer_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "xlmr":    ("xlm-roberta-base", "bilingual"),
    "mbert":   ("bert-base-multilingual-cased", "bilingual"),
    "arabert": ("aubmindlab/bert-base-arabertv2", "arabic"),
    "marbert": ("UBC-NLP/MARBERT", "arabic"),
}
if SMOKE_TEST:
    MODELS = {"smoke": ("prajjwal1/bert-tiny", "bilingual")}
    EPOCHS, LR_GRID, MAX_LEN = 1, [3e-5], 48

LABELS = ["Dissatisfied", "Satisfied"]     # index 0 = dissatisfied = positive class for recall / ROC
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def mount_drive_if_colab():
    if Path("/content").exists() and not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
mount_drive_if_colab()

train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "val.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")
if SMOKE_TEST:
    train, val, test = train.sample(300, random_state=SEED), val.sample(100, random_state=SEED), test.sample(150, random_state=SEED)
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tok):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=False)
        self.labels = [LABEL2ID[l] for l in labels]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

def collate(batch, pad_id):
    keys = [k for k in batch[0] if k != "labels"]
    maxlen = max(len(b["input_ids"]) for b in batch)
    out = {}
    for k in keys:
        fill = pad_id if k == "input_ids" else 0
        out[k] = torch.stack([torch.cat([b[k], torch.full((maxlen - len(b[k]),), fill, dtype=b[k].dtype)]) for b in batch])
    out["labels"] = torch.stack([b["labels"] for b in batch])
    return out

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        logits = model(**batch).logits
        probs.append(torch.softmax(logits, -1).cpu().numpy())
    return np.concatenate(probs)

def macro_f1(y_true_idx, probs):
    return f1_score(y_true_idx, probs.argmax(1), average="macro")

In [ ]:
def train_one(key, hf_name, lr, tr, va):
    seed_all()
    tok = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModelForSequenceClassification.from_pretrained(hf_name, num_labels=2).to(DEVICE)
    ds_tr = ReviewDataset(tr.review_text, tr.satisfaction_label, tok)
    ds_va = ReviewDataset(va.review_text, va.satisfaction_label, tok)
    pad = tok.pad_token_id
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, collate_fn=lambda b: collate(b, pad))
    dl_va = DataLoader(ds_va, batch_size=BATCH * 2, collate_fn=lambda b: collate(b, pad))

    # class-weighted loss: inverse frequency, normalised to mean 1
    counts = np.bincount(ds_tr.labels, minlength=2)
    w = torch.tensor(len(ds_tr.labels) / (2 * counts), dtype=torch.float).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = EPOCHS * len(dl_tr)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)

    best_f1, best_state, bad = -1, None, 0
    y_va = np.array(ds_va.labels)
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0
        for batch in dl_tr:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            tot += loss.item()
        f1 = macro_f1(y_va, predict(model, dl_va))
        print(f"    {key} lr={lr} epoch {epoch+1}: train loss {tot/len(dl_tr):.4f}  val macro-F1 {f1:.4f}  ({time.time()-t0:.0f}s)")
        if f1 > best_f1:
            best_f1, bad = f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print("    early stop"); break
    model.load_state_dict(best_state)
    return model, tok, best_f1

In [ ]:
runs = json.loads((OUT_DIR / "transformer_runs.json").read_text()) if (OUT_DIR / "transformer_runs.json").exists() else {}

for key, (hf_name, scope) in MODELS.items():
    if key in runs and not SMOKE_TEST:
        print(f"== {key}: already done, skipping"); continue
    print(f"\n== {key} ({hf_name}) on {scope} data")
    tr = train if scope == "bilingual" else train[train.language == "Arabic"]
    va = val if scope == "bilingual" else val[val.language == "Arabic"]
    te = test if scope == "bilingual" else test[test.language == "Arabic"]

    best = None
    for lr in LR_GRID:
        model, tok, f1 = train_one(key, hf_name, lr, tr, va)
        if best is None or f1 > best[2]:
            best = (model, tok, f1, lr)
        else:
            del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    model, tok, val_f1, lr = best

    ds_te = ReviewDataset(te.review_text, te.satisfaction_label, tok)
    dl_te = DataLoader(ds_te, batch_size=BATCH * 2, collate_fn=lambda b: collate(b, tok.pad_token_id))
    probs = predict(model, dl_te)
    pred = [LABELS[i] for i in probs.argmax(1)]
    test_f1 = f1_score(te.satisfaction_label, pred, average="macro")
    pd.DataFrame({"review_id": te.review_id.values, "y_true": te.satisfaction_label.values,
                  "y_pred": pred, "p_dissat": probs[:, LABEL2ID["Dissatisfied"]]}
                ).to_csv(OUT_DIR / f"{key}_test.csv", index=False)
    runs[key] = {"hf_model": hf_name, "scope": scope, "lr": lr, "max_len": MAX_LEN, "batch": BATCH,
                 "epochs_max": EPOCHS, "val_macro_f1": float(val_f1), "test_macro_f1": float(test_f1),
                 "n_train": int(len(tr)), "n_test": int(len(te)), "device": DEVICE}
    (OUT_DIR / "transformer_runs.json").write_text(json.dumps(runs, indent=2))
    print(f"   -> test macro-F1 {test_f1:.4f}  written {key}_test.csv")
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\noutputs in", OUT_DIR, sorted(p.name for p in OUT_DIR.iterdir()))

## Copying results back into the repository

```
results/predictions/xlmr_test.csv
results/predictions/mbert_test.csv
results/predictions/arabert_test.csv
results/predictions/marbert_test.csv
results/tables/transformer_runs.json
```

Then re-run `08_model_comparison.ipynb` and `10_segmentation.ipynb` (or `scripts/run_all.sh --post-colab`).